# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

# TODO : Enregistrer votre table Silver principale sous forme de vue temporaire SQL `v_silver`
SILVER_MAIN_PATH = "/lakehouse/silver/olist_order_items_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f79a527d-274d-4bad-bc3d-09cb94760c2b;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Lignes disponibles dans v_silver : 112650


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
# TODO : Calculer moyenne, écart-type, skewness et quantiles (P25, P50, P75, P90, P95) en Spark SQL

df_dist_summary = spark.sql("""
SELECT
    COUNT(*) AS nombre_lignes,
    AVG(item_total_amount) AS moyenne,
    STDDEV(item_total_amount) AS ecart_type,
    SKEWNESS(item_total_amount) AS skewness,
    PERCENTILE_APPROX(item_total_amount, 0.25) AS P25,
    PERCENTILE_APPROX(item_total_amount, 0.50) AS P50,
    PERCENTILE_APPROX(item_total_amount, 0.75) AS P75,
    PERCENTILE_APPROX(item_total_amount, 0.90) AS P90,
    PERCENTILE_APPROX(item_total_amount, 0.95) AS P95,
    MAX(item_total_amount) AS maximum
FROM v_silver
""")

df_dist_summary.show(truncate=False)


[Stage 13:===========================================>              (3 + 1) / 4]

+-------------+------------------+-----------------+-----------------+-----+----+------+------+------------------+-------+
|nombre_lignes|moyenne           |ecart_type       |skewness         |P25  |P50 |P75   |P90   |P95               |maximum|
+-------------+------------------+-----------------+-----------------+-----+----+------+------+------------------+-------+
|112650       |140.64405894362798|190.7243940317406|7.685984991648926|55.22|92.3|157.91|257.44|378.60999999999996|6929.31|
+-------------+------------------+-----------------+-----------------+-----+----+------+------+------------------+-------+



**Votre justification statistique *(À compléter)* :**
> `Expliquez ici ce que révèle le coefficient d'asymétrie (skewness) et l'écart entre la moyenne et la médiane sur vos données.`
> ### Justification statistique

Le coefficient d'asymétrie (Skewness = 7,68) montre que notre distribution présente une forte asymétrie positive. Cela signifie que la majorité des articles ont des montants relativement faibles, mais que quelques articles ont des montants très élevés.

On remarque également que la moyenne (140,64 €) est supérieure à la médiane (92,30 €). Cette différence montre que les valeurs extrêmes augmentent la moyenne.

Pour notre dashboard, nous choisissons donc principalement la médiane, car elle représente mieux le montant habituel des articles et est moins sensible aux valeurs extrêmes.

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:

GOLD_PARETO_PATH = "/lakehouse/gold/gold_pareto_order_status"

df_gold_1 = spark.sql("""
WITH aggregation AS (
    SELECT
        order_status,
        COUNT(*) AS nombre_articles,
        COUNT(DISTINCT order_id) AS nombre_commandes,
        SUM(item_total_amount) AS montant_total,
        PERCENTILE_APPROX(item_total_amount, 0.50) AS mediane,
        PERCENTILE_APPROX(item_total_amount, 0.75)
        - PERCENTILE_APPROX(item_total_amount, 0.25) AS iqr
    FROM v_silver
    GROUP BY order_status
),
pareto AS (
    SELECT
        *,
        DENSE_RANK() OVER (
            ORDER BY montant_total DESC
        ) AS classement,
        SUM(montant_total) OVER (
            ORDER BY montant_total DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / SUM(montant_total) OVER () AS pourcentage_cumule
    FROM aggregation
)
SELECT *
FROM pareto
ORDER BY classement
""")

df_gold_1.show(15, truncate=False)


26/11/17 15:50:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 15:50:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 15:50:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 15:50:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 15:50:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 15:51:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 1

+------------+---------------+----------------+--------------------+------------------+------------------+----------+------------------+
|order_status|nombre_articles|nombre_commandes|montant_total       |mediane           |iqr               |classement|pourcentage_cumule|
+------------+---------------+----------------+--------------------+------------------+------------------+----------+------------------+
|delivered   |110197         |96478           |1.5419773749999773E7|92.11000000000001 |102.28999999999999|1         |0.9732522443936316|
|shipped     |1185           |1106            |177129.3399999999   |95.02             |106.29000000000002|2         |0.9844321443388538|
|canceled    |542            |461             |105885.72           |97.8              |148.79000000000002|3         |0.9911153497029558|
|processing  |357            |301             |69394.11000000002   |124.5             |183.41            |4         |0.9954953084753858|
|invoiced    |359            |312        

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:

GOLD_TIME_PATH = "/lakehouse/gold/gold_monthly_evolution"

df_gold_2 = spark.sql("""
WITH monthly AS (
    SELECT
        DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS periode,
        SUM(item_total_amount) AS montant_total
    FROM v_silver
    GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
),
comparison AS (
    SELECT
        periode,
        montant_total,
        LAG(montant_total, 1) OVER (
            ORDER BY periode
        ) AS montant_precedent
    FROM monthly
)
SELECT
    periode,
    montant_total,
    montant_precedent,
    montant_total - montant_precedent AS evolution,
    CASE
        WHEN montant_precedent IS NOT NULL
             AND montant_precedent != 0
        THEN ROUND(
            (montant_total - montant_precedent)
            / montant_precedent * 100,
            2
        )
        ELSE NULL
    END AS taux_evolution_pct
FROM comparison
ORDER BY periode
""")

# Sauvegarder la table Gold en Delta
df_gold_2.coalesce(1) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_TIME_PATH)

print("Table Gold temporelle sauvegardée avec succès !")

df_gold_2.show(20, truncate=False)


26/11/17 16:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
          

Table Gold temporelle sauvegardée avec succès !


26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 1

+-------+------------------+------------------+-------------------+------------------+
|periode|montant_total     |montant_precedent |evolution          |taux_evolution_pct|
+-------+------------------+------------------+-------------------+------------------+
|2016-09|354.75            |NULL              |NULL               |NULL              |
|2016-10|56808.84          |354.75            |56454.09           |15913.77          |
|2016-12|19.62             |56808.84          |-56789.219999999994|-99.97            |
|2017-01|137188.49000000002|19.62             |137168.87000000002 |699127.78         |
|2017-02|286280.6199999999 |137188.49000000002|149092.12999999986 |108.68            |
|2017-03|432048.58999999973|286280.6199999999 |145767.96999999986 |50.92             |
|2017-04|412422.23999999976|432048.58999999973|-19626.349999999977|-4.54             |
|2017-05|586190.9499999993 |412422.23999999976|173768.7099999995  |42.13             |
|2017-06|502963.0400000001 |586190.94999999

In [4]:

GOLD_TIME_PATH = "/lakehouse/gold/gold_monthly_evolution"

df_gold_2 = spark.sql("""
WITH monthly AS (
    SELECT
        DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS periode,
        SUM(item_total_amount) AS montant_total
    FROM v_silver
    GROUP BY DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
),
comparison AS (
    SELECT
        periode,
        montant_total,
        LAG(montant_total, 1) OVER (
            ORDER BY periode
        ) AS montant_precedent
    FROM monthly
)
SELECT
    periode,
    montant_total,
    montant_precedent,
    montant_total - montant_precedent AS evolution,
    CASE
        WHEN montant_precedent IS NOT NULL
             AND montant_precedent != 0
        THEN ROUND(
            (montant_total - montant_precedent)
            / montant_precedent * 100,
            2
        )
        ELSE NULL
    END AS taux_evolution_pct
FROM comparison
ORDER BY periode
""")

# Sauvegarder la table Gold en Delta
df_gold_2.coalesce(1) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_TIME_PATH)

print("Table Gold temporelle sauvegardée avec succès !")

df_gold_2.show(20, truncate=False)


26/11/17 16:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:12:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
          

Table Gold temporelle sauvegardée avec succès !


26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 16:14:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/17 1

+-------+------------------+------------------+-------------------+------------------+
|periode|montant_total     |montant_precedent |evolution          |taux_evolution_pct|
+-------+------------------+------------------+-------------------+------------------+
|2016-09|354.75            |NULL              |NULL               |NULL              |
|2016-10|56808.84          |354.75            |56454.09           |15913.77          |
|2016-12|19.62             |56808.84          |-56789.219999999994|-99.97            |
|2017-01|137188.49000000002|19.62             |137168.87000000002 |699127.78         |
|2017-02|286280.6199999999 |137188.49000000002|149092.12999999986 |108.68            |
|2017-03|432048.58999999973|286280.6199999999 |145767.96999999986 |50.92             |
|2017-04|412422.23999999976|432048.58999999973|-19626.349999999977|-4.54             |
|2017-05|586190.9499999993 |412422.23999999976|173768.7099999995  |42.13             |
|2017-06|502963.0400000001 |586190.94999999

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
# TODO : Créer votre 3e table Gold (optionnel mais recommandé pour enrichir le dashboard) + lancer OPTIMIZE sur Silver
GOLD_ANALYSIS_PATH = "/lakehouse/gold/gold_top_month_by_status"

df_gold_3 = spark.sql("""
WITH aggregation_mensuelle AS (
    SELECT
        order_status,
        DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM') AS periode,
        SUM(item_total_amount) AS montant_total
    FROM v_silver
    GROUP BY
        order_status,
        DATE_FORMAT(order_purchase_timestamp, 'yyyy-MM')
),
classement AS (
    SELECT
        order_status,
        periode,
        montant_total,
        ROW_NUMBER() OVER (
            PARTITION BY order_status
            ORDER BY montant_total DESC
        ) AS rang
    FROM aggregation_mensuelle
)
SELECT
    order_status,
    periode,
    montant_total
FROM classement
WHERE rang = 1
ORDER BY montant_total DESC
""")

df_gold_3.show(truncate=False)

df_gold_3.coalesce(1) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .save(GOLD_ANALYSIS_PATH)

print("Troisième table Gold sauvegardée avec succès !")

+------------+-------+------------------+
|order_status|periode|montant_total     |
+------------+-------+------------------+
|delivered   |2017-11|1153364.2000000011|
|shipped     |2018-03|20620.66          |
|canceled    |2018-07|18944.45          |
|invoiced    |2018-03|9170.44           |
|processing  |2017-12|8842.33           |
|unavailable |2016-10|2140.49           |
|approved    |2017-04|179.46            |
+------------+-------+------------------+



Troisième table Gold sauvegardée avec succès !


In [3]:
spark.sql("DESCRIBE v_silver").show(truncate=False)

+-----------------------------+---------+-------+
|col_name                     |data_type|comment|
+-----------------------------+---------+-------+
|order_id                     |string   |NULL   |
|order_item_id                |int      |NULL   |
|product_id                   |string   |NULL   |
|seller_id                    |string   |NULL   |
|shipping_limit_date          |timestamp|NULL   |
|price                        |double   |NULL   |
|freight_value                |double   |NULL   |
|_ingested_at                 |timestamp|NULL   |
|_source_dataset              |string   |NULL   |
|_source_file                 |string   |NULL   |
|customer_id                  |string   |NULL   |
|order_status                 |string   |NULL   |
|order_purchase_timestamp     |timestamp|NULL   |
|order_delivered_customer_date|timestamp|NULL   |
|item_total_amount            |double   |NULL   |
|delivery_delay_days          |int      |NULL   |
+-----------------------------+---------+-------+


In [5]:
print(spark)

In [2]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [6]:
spark.sql(f"""
OPTIMIZE delta.`{SILVER_MAIN_PATH}`
""").show(truncate=False)

+----------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                            |metrics                                                                                                                                                                                |
+----------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/olist_order_items_enriched|{1, 4, {13344348, 13344348, 1.3344348E7, 1, 13344348}, {3458213, 3643193, 3518048.75, 4, 14072195}, 1, NULL, NULL, 1, 1, 4, 0, false, 0, 0, 1794932530042, 0, 4, 0, NULL, NULL, 16, 1